In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('course_lead_scoring_2026.csv')

print(df.shape)
df.head()

(5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [2]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [3]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [4]:
categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [5]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [6]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

In [7]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [8]:
numerical_features = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

corr = df[numerical_features].corr()

corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [9]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for a, b in pairs:
    print(a, 'and', b, ':', corr.loc[a, b])

interaction_count and lead_score : 0.9157457980418697
number_of_courses_viewed and lead_score : 0.7572042392104886
number_of_courses_viewed and interaction_count : 0.7216090038937035
annual_income and interaction_count : 0.12284235135955958


In [10]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=42
)

In [11]:
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

df_train = df_train.drop(columns=['converted'])
df_val = df_val.drop(columns=['converted'])
df_test = df_test.drop(columns=['converted'])

In [12]:
df_train.shape, df_val.shape, df_test.shape

((3000, 8), (1000, 8), (1000, 8))

In [13]:
'converted' in df_train.columns

False

In [14]:

from sklearn.metrics import mutual_info_score

categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mutual_info_converted_score)

mi.sort_values(ascending=False).round(2)


lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [15]:

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Convert dataframes into dictionaries
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

# One-hot encode categorical features
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# Train logistic regression
model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print("Validation accuracy:", accuracy)
print("Rounded accuracy:", round(accuracy, 2))


Validation accuracy: 0.645
Rounded accuracy: 0.65


In [16]:

features_to_remove = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

baseline_accuracy = accuracy_score(y_val, model.predict(X_val))

for feature in features_to_remove:
    # Remove one feature from both datasets
    train_reduced = df_train.drop(columns=[feature])
    val_reduced = df_val.drop(columns=[feature])

    # Encode using the reduced training features
    dv_reduced = DictVectorizer()
    X_train_reduced = dv_reduced.fit_transform(
        train_reduced.to_dict(orient='records')
    )
    X_val_reduced = dv_reduced.transform(
        val_reduced.to_dict(orient='records')
    )

    # Train the same model
    model_reduced = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )
    model_reduced.fit(X_train_reduced, y_train)

    # Compare validation accuracy
    reduced_accuracy = accuracy_score(
        y_val, model_reduced.predict(X_val_reduced)
    )

    difference = baseline_accuracy - reduced_accuracy

    print(
        feature,
        "accuracy:", round(reduced_accuracy, 3),
        "difference:", round(difference, 3)
    )


lead_source accuracy: 0.642 difference: 0.003
number_of_courses_viewed accuracy: 0.643 difference: 0.002
interaction_count accuracy: 0.601 difference: 0.044


In [17]:

C_values = [0.000001, 0.00001, 0.0001, 0.001]

results = []

for C in C_values:
    model_reg = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )

    model_reg.fit(X_train, y_train)

    y_pred_reg = model_reg.predict(X_val)
    accuracy_reg = accuracy_score(y_val, y_pred_reg)

    results.append((C, accuracy_reg))

    print(f"C={C}, accuracy={accuracy_reg:.3f}")

# Best accuracy; if tied, choose the smallest C
best_C, best_accuracy = sorted(
    results,
    key=lambda item: (-item[1], item[0])
)[0]

print("Best C:", best_C)
print("Best accuracy:", round(best_accuracy, 3))


C=1e-06, accuracy=0.598
C=1e-05, accuracy=0.598
C=0.0001, accuracy=0.613
C=0.001, accuracy=0.645
Best C: 0.001
Best accuracy: 0.645
